# FI-Ran clear-cut: diagnosis and fix of three convergence issues

**Meeting summary -- branch `debug/tl-convergence-isolation`**

**TL;DR:** In the FI-Ran clear-cut simulation (Ranskalankorpi, June 2022), three nested
iterative solves did not always converge within their iteration budget: the canopy's outer
Picard loop (`mlm_canopy`), the wet-leaf energy balance in rainfall interception
(`interception`), and the dry-leaf gas exchange loop (`planttype`). Relaxation (gamma
damping) was added to all three. The full June 2022 run (1392 timesteps, 30 days) now
converges **100% in all three modules** -- a longer historical log had recorded 1147
non-convergence events in total.

## Background: what was broken

`CanopyModel.run()` (`pyAPES/canopy/mlm_canopy.py`) solves the canopy's temperature, H2O
and CO2 profiles iteratively (Picard loop) at every timestep, and two other modules solve
their own energy balance on their own nested loop inside that:

- `Interception.run()` -- wet-leaf temperature (rainfall evaporation)
- `PlantType.leaf_gas_exchange()` -- dry-leaf temperature (photosynthesis + transpiration),
  solved separately for each of 3 planttypes (spruce/decid/shrubs) and both leaf types
  (sunlit/shaded)

When any of these failed to converge within its iteration budget, the model either
continued with an inexact value ("tolerable") or fell back to a coarser well-mixed
assumption ("switched to WMA"). `Examples/logs/ran_22_k7.log` (a longer historical run)
shows how often this happened:

In [ ]:
import os
import sys

assert os.path.basename(os.getcwd()) == 'Examples', (
    f"expected to run with cwd=Examples/, got {os.getcwd()!r} -- adjust paths below if not")
sys.path.insert(0, os.path.abspath('..'))

import re
import pickle
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

COLOR_CRITICAL = '#d03b3b'   # validated status palette (dataviz skill)
COLOR_GOOD = '#0ca30c'
MODULE_ORDER = ['mlm_canopy', 'interception', 'planttype']
MODULE_LABELS = {
    'mlm_canopy': 'mlm_canopy\n(outer Picard loop)',
    'interception': 'interception\n(wet-leaf loop)',
    'planttype': 'planttype\n(leaf_gas_exchange)',
}


def count_debug_messages(log_path, module_map):
    counts = Counter()
    pattern = re.compile(r'DEBUG (pyAPES\.\S+) ')
    with open(log_path) as f:
        for line in f:
            m = pattern.match(line)
            if not m:
                continue
            for key, needle in module_map.items():
                if needle in m.group(1):
                    counts[key] += 1
    return counts


historical_counts = count_debug_messages('logs/ran_22_k7.log', {
    'mlm_canopy': 'canopy.mlm_canopy',
    'interception': 'canopy.interception',
    'planttype': 'planttype.planttype',
})
print('Historical log (Examples/logs/ran_22_k7.log, longer period):')
for m in MODULE_ORDER:
    print(f'  {m}: {historical_counts[m]}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
values = [historical_counts[m] for m in MODULE_ORDER]
bars = ax.bar([MODULE_LABELS[m] for m in MODULE_ORDER], values, color=COLOR_CRITICAL, width=0.6)
for bar, v in zip(bars, values):
    ax.annotate(str(v), (bar.get_x() + bar.get_width() / 2, v), ha='center', va='bottom', fontsize=11)
ax.set_ylabel('convergence DEBUG messages (count)')
ax.set_title('Original diagnosis: non-convergence by module')
for spine in ('top', 'right'):
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.show()

## Three problems found and their fixes

| # | module | problem | fix |
|---|---|---|---|
| 1 | `mlm_canopy.run()` (`pyAPES/canopy/mlm_canopy.py`) | the outer loop's relaxation factor `gam` couldn't drop low enough in oscillating cases | lowered `gam_floor` from `0.25` to `0.01` |
| 2 | `Interception.run()` (`pyAPES/canopy/interception.py`) | wet-leaf temperature loop had no relaxation at all (plain fixed-point) | added oscillation-adaptive gamma relaxation (`gamma=0.75` start, `gamma_floor=0.01`) |
| 3 | `PlantType.leaf_gas_exchange()` (`pyAPES/planttype/planttype.py`) | dry-leaf temperature loop had no relaxation at all, same issue as interception | added the same oscillation-adaptive relaxation (`gamma=1.0` start, `gamma_floor=0.05`) |

Each fix was first validated in a standalone sandbox notebook (a copy of just the loop,
run against already-captured real forcing data, so candidate values could be tried without
re-running the full source code each time):

- `debug_mlm_canopy_convergence.ipynb` -- 19 captured non-convergent timesteps,
  `gam0=0.1, gam_floor=0.15`: **18/19 converge** (vs. the `gam_floor=0.01` that ended up in source)
- `debug_interception_relaxation.ipynb` -- 1392 timesteps (June 2022):
  baseline (no relaxation) **3/1392 fail**; `gamma=0.5`: **0/1392 fail**
  (but mean iteration count roughly doubles across the whole dataset)
- `debug_planttype_leaf_temperature_relaxation.ipynb` -- 8352 combinations
  (1392 timesteps x 3 planttypes x 2 leaf types): baseline **11/8352 fail**
  (all `spruce`/`sunlit`, midday hours); oscillation-adaptive relaxation:
  **0/8352 fail**, mean iteration count **essentially unchanged** (3.77 -> 3.76)

**Two implementation bugs were also found and fixed along the way** that would otherwise
have prevented the fixes from working at all: in `interception.py` the relaxation formula
referenced a mismatched array shape (`ValueError: shapes (7,) (41,)`), and in `planttype.py`
the oscillation branch called `np.max(a, b)` instead of `max(a, b)` (`TypeError`, crashed
the full run about 80% through the June simulation).

## Result: full June 2022 run with all three fixes

The run below was executed specifically for this notebook
(`PYTHONPATH=. .venv/bin/python Examples/capture_forcing_exploration.py`,
1.-30.6.2022, 1392 timesteps, 30-min timestep) against the current, fixed source code --
not simulated, a real run.

In [ ]:
current_counts = count_debug_messages('../logs/capture_forcing_exploration.log', {
    'mlm_canopy': 'canopy.mlm_canopy',
    'interception': 'canopy.interception',
    'planttype': 'planttype.planttype',
})

outcomes = {}
for tag in MODULE_ORDER:
    with open(f'debug_captures/forcing_exploration/{tag}_forcing_samples.pkl', 'rb') as f:
        recs = pickle.load(f)
    outcomes[tag] = Counter(r['outcome'] for r in recs)

print('Current run (June 2022, all 3 fixes):')
for m in MODULE_ORDER:
    print(f'  {m}: {current_counts[m]} DEBUG messages')
print()
print('Outer-loop (mlm_canopy) outcome, 1392 timesteps:')
print(' ', dict(outcomes['mlm_canopy']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# --- left: per-module debug-message count, before (historical) vs after (now) ---
ax = axes[0]
x = np.arange(len(MODULE_ORDER))
width = 0.35
before_vals = [historical_counts[m] for m in MODULE_ORDER]
after_vals = [current_counts[m] for m in MODULE_ORDER]
ax.bar(x - width / 2, before_vals, width, color=COLOR_CRITICAL, label='before (historical log)')
ax.bar(x + width / 2, after_vals, width, color=COLOR_GOOD, label='now (June, fixed)')
for xi, v in zip(x - width / 2, before_vals):
    ax.annotate(str(v), (xi, v), ha='center', va='bottom', fontsize=9)
for xi, v in zip(x + width / 2, after_vals):
    ax.annotate(str(v), (xi, v), ha='center', va='bottom', fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels([MODULE_LABELS[m] for m in MODULE_ORDER])
ax.set_ylabel('convergence DEBUG messages (count)')
ax.set_title('DEBUG messages per module: before vs now')
ax.legend(fontsize=8)
for spine in ('top', 'right'):
    ax.spines[spine].set_visible(False)

# --- right: outer-loop outcome, before (historical, documented earlier in this research
#     session from a June run made before the interception/planttype fixes) vs now
#     (just executed, all 3 fixes) ---
ax = axes[1]
outcome_labels = ['converged', 'tolerable', 'switched_to_wma']
# earlier June run (only the outer-loop gam_floor fix applied, interception/planttype
# not yet fixed) -- observed and recorded earlier in this research session
before_outcome = {'converged': 1308, 'tolerable': 74, 'switched_to_wma': 10}
after_outcome = {k: outcomes['mlm_canopy'].get(k, 0) for k in outcome_labels}
before_v = [before_outcome[k] for k in outcome_labels]
after_v = [after_outcome[k] for k in outcome_labels]
x2 = np.arange(len(outcome_labels))
ax.bar(x2 - width / 2, before_v, width, color=COLOR_CRITICAL, label='before (interception/planttype not yet fixed)')
ax.bar(x2 + width / 2, after_v, width, color=COLOR_GOOD, label='now (all 3 fixes)')
for xi, v in zip(x2 - width / 2, before_v):
    if v:
        ax.annotate(str(v), (xi, v), ha='center', va='bottom', fontsize=9)
for xi, v in zip(x2 + width / 2, after_v):
    if v:
        ax.annotate(str(v), (xi, v), ha='center', va='bottom', fontsize=9)
ax.set_xticks(x2)
ax.set_xticklabels(outcome_labels, rotation=15)
ax.set_ylabel('timesteps (June, 1392 total)')
ax.set_title('mlm_canopy outer-loop outcome: before vs now')
ax.legend(fontsize=8)
for spine in ('top', 'right'):
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.show()

## Notes and caveats

- **The left chart isn't an exact apples-to-apples period**: the historical log
  (`ran_22_k7.log`) spans a longer, multi-month window, while the "now" bars are for one
  June -- the order of magnitude is nonetheless unambiguous (627 -> 0, 337 -> 0, 183 -> 7).
- **The right chart** compares two June runs directly (same 1392 timesteps, same forcing),
  so it's a genuine before/after: the outer loop's 74 tolerable + 10 switched-to-WMA
  timesteps disappeared entirely.
- The remaining 7 `interception` DEBUG messages are just notices (`itermax` reached, but
  error still small, 0.04-0.20) -- they didn't prevent outer-loop convergence in any of the
  1392 timesteps.
- This branch (`debug/tl-convergence-isolation`) is a research branch. The debug-capture
  instrumentation (`pyAPES/utils/debug_capture.py`, hooks in `mlm_canopy.py`) is opt-in and
  has no effect on normal runs, but it and this notebook and the other
  `Examples/debug_*` files should be cleaned up or dropped before any merge to `main`.

## Next steps

1. A light code review of the three changes (`mlm_canopy.py`, `interception.py`,
   `planttype.py`) before considering them for further use.
2. A longer validation run (e.g. a full growing season or multiple years) to confirm the
   result holds outside of June as well.
3. Clean up / remove the debug instrumentation (`debug_capture.py`) and research notebooks
   before merging this branch into `main`, if it gets merged.